In [19]:
# Import required modules
import numpy as np
import os
from pathlib import Path
import pandas as pd
import skimage
import dask.array as da
import dask
from acid.utils.listdirNHF import listdirNHF
from acid.image_processing.rescale_intensity import quantize_image
from acid.feature_extraction.measure_haralick import measure_haralick_features, glcm_feature_map


# boundary should be set to none for glcm feature map

# note the rescaling of the image intensities

In [2]:
from dask.distributed import Client
client = Client()
print(client)

<Client: 'tcp://127.0.0.1:54552' processes=11 threads=22, memory=31.46 GiB>


In [3]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 11
Total threads: 22,Total memory: 31.46 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54552,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:54608,Total threads: 2
Dashboard: http://127.0.0.1:54620/status,Memory: 2.86 GiB
Nanny: tcp://127.0.0.1:54555,


In [16]:
input_dir_image = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\260326_parallelization_strategy\fov_proc"
input_dir_mask = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\260326_parallelization_strategy\seg"
filenames = listdirNHF(Path(input_dir_image))


In [17]:
test_file_name = filenames[0]
test_file = skimage.io.imread(os.path.join(Path(input_dir_image), test_file_name))
print(test_file.shape)

(5, 1024, 1024)


In [ ]:
@dask.delayed
def load(filename,
         input_dir_image=input_dir_image,
         input_dir_mask=input_dir_mask):
    """
    needs to return an array. Expects filename to be the same for
    the image and segmentation mask.
    """
    image = skimage.io.imread(os.path.join(Path(input_dir_image), filename))
    mask = skimage.io.imread(os.path.join(Path(input_dir_mask), filename))

    return image, mask


@dask.delayed
def process_image_mask_mbfm(image, mask, input_channel_axis=-1, levels=8):
    """
    needs to receive an array and return an array.
    Expects the mask to be concatenated to the image data along the channel axis.
    
    This function is meant to prepare image and mask for the mask-based computation
    of the feature map, by concatenating the mask to the image data along the channel axis.
    """
    rescaled_image = []
    for c in np.unstack(image,axis=input_channel_axis):
        # introduce a very mild gaussian smoothing

        rescaled_channel = quantize_image(c,
                                          levels=levels)
        
        rescaled_image.append(rescaled_channel)
    
    rescaled_image = np.stack(rescaled_image, axis=input_channel_axis)
    mask = np.expand_dims(mask, axis=input_channel_axis) # add a channel axis to the mask if it doesn't have one
    data = np.concatenate([rescaled_image, mask], axis=input_channel_axis)
    
    # ensure the correct order of the axes (channel axis in the last position)
    if input_channel_axis != -1:
        data = np.moveaxis(data, input_channel_axis, -1)
    
    return data, mask


@dask.delayed
def compute_mask_based_feature_map(data,
                                   feature_funct,
                                   window_size=11,
                                   chunks=(256, 256, -1),
                                   boundary='reflect',
                                   dtype=float,
                                   trim=True,
                                   kwargs=None):
    """
    needs to receive an array and return an array.
    Expects channel axis in the last position.
    Expects the mask to be concatenated to the image data as the last sub-stack
    of the channel axis.

    feature_funct must be a top-level function that takes as input an image and a mask, and returns a feature map of the same spatial dimensions as the input image.
    """
    if kwargs is None:
        kwargs = {}
    
    depth = {0: window_size // 2, 1: window_size // 2, 2: 0} # depth for the spatial dimensions
    
    data_da = da.from_array(data, chunks=chunks) # chunks in xy but not on the channel axis

    def _wrapper(data_block, **w_kwargs):
        # data_block is a numpy array corresponding to a chunk of the input data
        # it has the same structure as the input data (channel axis in the last position, mask concatenated to image data)
        
        img_block = data_block[..., :-1] # all channels except the last one are image channels
        mask_block = data_block[..., -1] # the last channel is the mask

        return feature_funct(image=img_block,
                                mask=mask_block,
                                **w_kwargs)

    fm = data_da.map_overlap(_wrapper,
                             depth=depth,
                             boundary=boundary,
                             dtype=dtype,
                             trim=trim,
                             **kwargs) # trim the overlapping regions after computation to avoid double-counting

    return fm.compute() # returns a numpy array


@dask.delayed
def measure_feature_map(feature_map, mask):
    # generates measurements from the feature map and the mask
    return # returns a dataframe of measurements


def f_mbfm(filenames,
           feature_funct,
           channel_axis=-1,
           levels=8,
           window_size=11,
           chunks=(256, 256, -1),
           boundary='reflect',
           dtype=float,
           trip=True,
           kwargs=None):
    """
    
    """
    if kwargs is None:
        kwargs = {}

    results = []
    
    for filename in filenames:
        
        image, mask = load(filename)
        
        data, mask = process_image_mask_mbfm(image,
                                             mask,
                                             input_channel_axis=channel_axis,
                                             levels=levels)
        
        fm = compute_mask_based_feature_map(data,
                                            feature_funct,
                                            window_size=11,
                                   chunks=(256, 256, -1),
                                   boundary='reflect',
                                   dtype=float,
                                   trim=True,
                                            **kwargs)
        
        measurements = measure_feature_map(fm,mask)
        
        results.append(measurements)

    return results




In [ ]:
channel_axis=1
levels = 8
window_size=11
boundary='none' # none is used for haralick features because the image (and image chunks) are already padded within glcm_feature_map_ch
chunks=(256, 256, -1)
glcm_kwargs = {
'props':None,
'distances':None,
'angles':None,
'mask':None,
'channel_axis':None,
'window_shape':11,
'graycomtx_kwargs':{'levels':levels, 'symmetric':True,'normed':True},
'pad_kwargs':None,
'windows_kwargs':None,
'zeros_kwargs':None,
'glcm_concat_kwargs':None,
'stack_channels':True,
'stack_axis':-1,
'stack_kwargs':None,
'feature_concat_axis':-1,
'feature_concat_kwargs':None
}





In [ ]:
# # Indicate the path to the input image
# input_image___path = os.path.join(os.getcwd(),"secondary_output")

# # get the name of the input image
# input_image___name = "260202_fov_example.ome.tif"

# # open input image
# input_real_image = io.imread(os.path.join(input_image___path, input_image___name))
# print(input_real_image.shape)

# # the first image of the channel axis is the segmentation mask. The following images are channels, to be analysed


(2720, 2720, 7)


the file is a 2720x2720 field of view with 7 channels.

The first of the 7 channels is the segmentation mask.

Objects are individual cells.

The remaining 6 channels are different imaged structures / imaging modalities.

In [ ]:
# real_data_haralick_features = measure_haralick_features(image=input_real_image[...,1:3],
#                                              label_image=input_real_image[...,0],
#                                              props=['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'],
#                                              distances=[5, 15, 49],
#                                              angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
#                                              channel_axis=-1,
#                                              window_shape=50,
#                                              glcm_daskbag_kwargs={'npartitions': 10})

# real_data_haralick_features

# # single channel,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 10 partitions -> 8m 16s
# # double channels,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 12 partitions -> 15m 33s



Default: Rescaling image to 8 intensity levels - indicate levels in glcm_graycomtx_kwargs to avoid this
